In [20]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

In [21]:
# Path to dataset
DATA_PATH = "../data/DataCoSupplyChainDataset.csv"

df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset Loaded Successfully")
print("Shape:", df.shape)
df.head()

Dataset Loaded Successfully
Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [22]:
columns_to_drop = [
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",
    "Customer Street",
    "Customer Zipcode",
    "Order Zipcode",
    "Product Description",
    "Product Image"
]

df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

print("Columns after cleaning:")
print(df.columns.tolist())


Columns after cleaning:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']


In [23]:
print("Missing values per column:")
print(df.isnull().sum())

# Fill numeric nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical nulls with mode
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values handled.")

Missing values per column:
Type                             0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Benefit per order                0
Sales per customer               0
Delivery Status                  0
Late_delivery_risk               0
Category Id                      0
Category Name                    0
Customer City                    0
Customer Country                 0
Customer Id                      0
Customer Segment                 0
Customer State                   0
Department Id                    0
Department Name                  0
Latitude                         0
Longitude                        0
Market                           0
Order City                       0
Order Country                    0
Order Customer Id                0
order date (DateOrders)          0
Order Id                         0
Order Item Cardprod Id           0
Order Item Discount              0
Order Item Discount Rate         0
Order Item Id               

C:\Users\srich\AppData\Local\Temp\ipykernel_23500\1390918629.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns
C:\Users\srich\AppData\Local\Temp\ipykernel_23500\1390918629.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, 

Missing values handled.


In [24]:
df["order_date"] = pd.to_datetime(df["order date (DateOrders)"])
df["shipping_date"] = pd.to_datetime(df["shipping date (DateOrders)"])

df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["day"] = df["order_date"].dt.day
df["day_of_week"] = df["order_date"].dt.dayofweek

In [25]:
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # winter
    elif month in [3, 4, 5]:
        return 1  # spring
    elif month in [6, 7, 8]:
        return 2  # summer
    else:
        return 3  # fall

df["season"] = df["month"].apply(get_season)

In [26]:
df["delay_days"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
df["is_delayed"] = (df["delay_days"] > 0).astype(int)

df["lead_time"] = df["Days for shipping (real)"]

In [27]:
np.random.seed(42)

# Simulate route distance between 100 km and 15000 km
df["distance_km"] = np.random.uniform(100, 15000, len(df))

In [28]:
country_risk_map = {
    "India": 0.45,
    "Indonesia": 0.50,
    "Australia": 0.20,
    "Puerto Rico": 0.35,
    "United States": 0.25
}

df["political_risk_origin"] = df["Customer Country"].map(country_risk_map).fillna(0.40)
df["political_risk_destination"] = df["Order Country"].map(country_risk_map).fillna(0.40)

df["political_risk"] = (
    df["political_risk_origin"] + df["political_risk_destination"]
) / 2
df["is_international"] = (
    df["Customer Country"] != df["Order Country"]
).astype(int)


In [29]:
season_weather_risk = {
    0: 0.6,  # winter
    1: 0.3,  # spring
    2: 0.4,  # summer
    3: 0.35  # fall
}

df["weather_risk"] = df["season"].map(season_weather_risk)

In [30]:
transport_risk_map = {
    "Standard Class": 0.4,
    "First Class": 0.2,
    "Second Class": 0.3,
    "Same Day": 0.1
}

df["transport_risk"] = df["Shipping Mode"].map(transport_risk_map).fillna(0.35)


In [31]:
demand_df = (
    df.groupby(["Product Card Id", "order_date"])["Order Item Quantity"]
    .sum()
    .reset_index()
)

demand_df.rename(columns={"Order Item Quantity": "daily_demand"}, inplace=True)


In [32]:
demand_df["rolling_7"] = (
    demand_df.groupby("Product Card Id")["daily_demand"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

In [33]:
df["simulated_inventory"] = np.random.randint(50, 500, len(df))

df["stockout_risk"] = (
    df["Order Item Quantity"] > df["simulated_inventory"]
).astype(int)

df["overstock_risk"] = (
    df["simulated_inventory"] > df["Order Item Quantity"] * 3
).astype(int)

In [34]:
df["resilience_score"] = (
    100
    - (df["Late_delivery_risk"] * 30)
    - (df["political_risk"] * 20)
    - (df["weather_risk"] * 15)
    - (df["stockout_risk"] * 25)
    - (df["distance_km"] / 15000 * 10)
)

df["resilience_score"] = df["resilience_score"].clip(0, 100)

In [37]:
model_features = [
    "Product Card Id",
    "Order Item Quantity",
    "distance_km",
    "political_risk",
    "weather_risk",
    "transport_risk",
    "month",
    "season",
    "is_international",
    "stockout_risk",
    "overstock_risk",
    "Late_delivery_risk",
    "resilience_score"
]

processed_df = df[model_features]

print("Final processed dataset shape:", processed_df.shape)
processed_df.head()

Final processed dataset shape: (180519, 13)


,Product Card Id,Order Item Quantity,distance_km,political_risk,weather_risk,transport_risk,month,season,is_international,stockout_risk,overstock_risk,Late_delivery_risk,resilience_score
0,1360,1,5680.647771,0.425,0.6,0.4,1,0,1,0,1,0,78.712901
1,1360,1,14265.643166,0.400,0.6,0.4,1,0,1,0,1,1,43.489571
2,1360,1,11006.709733,0.425,0.6,0.4,1,0,1,0,1,0,75.162194
3,1360,1,9020.011415,0.300,0.6,0.4,1,0,1,0,1,0,78.986659
4,1360,1,2424.677743,0.275,0.6,0.4,1,0,1,0,1,0,83.883548


In [38]:
OUTPUT_PATH = "../data/processed_supply_chain.csv"
processed_df.to_csv(OUTPUT_PATH, index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
